# OME-Zarr Writer — Plate Layout

Compare with `ome_zarr_writer.ipynb` which uses `OmeZarrWriter` (position slider).

This notebook uses `OmeZarrWriterPlate` instead, which stores each FOV
as a well in an OME-Zarr plate. napari-ome-zarr tiles the positions
spatially as a mosaic.

| | `OmeZarrWriter` | `OmeZarrWriterPlate` |
|--|--|--|
| Positions in napari | **slider** (p dimension) | **spatial mosaic** |
| Array layout | single 5D `(t, p, c, y, x)` | per-well `(t, c, y, x)` |
| Labels in napari | yes | **no** (reader limitation) |

> **Note:** Labels (segmentation masks, tracked labels, stim masks) ARE
> written into each well's zarr group and can be loaded programmatically.
> However, `napari-ome-zarr`'s plate/well reader does not descend into
> wells to discover label sub-groups — it only loads the raw image mosaic.
> This is a limitation of `ome-zarr-py`'s `Well` spec, not of the data format.

## 1. Connect to microscope

In [1]:
from faro.microscope.demo import MMDemo

mic = MMDemo(micromanager_path="C:\\Program Files\\Micro-Manager-2.0")

In [2]:
import faro.core.utils as utils

utils.print_configs(mic.mmc)

mic.mmc.snapImage()
test_img = mic.mmc.getImage()
print(f"Camera: {test_img.shape[1]}x{test_img.shape[0]}, dtype={test_img.dtype}")

Config Groups
├── Camera
│   ├── HighRes
│   ├── LowRes
│   └── MedRes
├── Channel
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── Channel-Multiband
│   ├── Cy5
│   ├── DAPI
│   ├── FITC
│   └── Rhodamine
├── LightPath
│   ├── Camera-left
│   ├── Camera-right
│   └── Eyepiece
├── Objective
│   ├── 10X
│   ├── 20X
│   └── 40X
└── System
    └── Startup

Camera: 512x512, dtype=uint16


## 2. Set up the pipeline

In [3]:
import os
import shutil
import tempfile

from faro.core.data_structures import SegmentationMethod
from faro.segmentation.base import OtsuSegmentator
from faro.tracking.trackpy import TrackerTrackpy
from faro.feature_extraction.simple import SimpleFE
from faro.core.pipeline import ImageProcessingPipeline
from faro.stimulation.base import StimWholeFOV

path = os.path.join(tempfile.gettempdir(), "rtm-ome-zarr-plate-test")
if os.path.exists(path):
    shutil.rmtree(path)

segmentators = [
    SegmentationMethod(
        name="labels",
        segmentation_class=OtsuSegmentator(),
        use_channel=0,
        save_tracked=True,
    )
]

pipeline = ImageProcessingPipeline(
    storage_path=path,
    segmentators=segmentators,
    feature_extractor=SimpleFE("labels"),
    tracker=TrackerTrackpy(),
    stimulator=StimWholeFOV(),
)

Directory C:\Users\Alex\AppData\Local\Temp\rtm-ome-zarr-plate-test\tracks created 


## 3. Create the OmeZarrWriterPlate

The only difference from the slider notebook: `OmeZarrWriterPlate` instead of `OmeZarrWriter`.

In [4]:
from faro.core.writers import OmeZarrWriterPlate

writer = OmeZarrWriterPlate(
    storage_path=path,
    store_stim_images=True,
)

print(f"Zarr store: {writer._zarr_path}")

Zarr store: C:\Users\Alex\AppData\Local\Temp\rtm-ome-zarr-plate-test\acquisition.ome.zarr


## 4. Define experiment (same as slider notebook)

In [5]:
from faro.core.data_structures import RTMSequence

seq = RTMSequence(
    time_plan={"interval": 0.5, "loops": 4},
    stage_positions=[
        {"x": 0.0, "y": 0.0, "z": 0.0},
        {"x": 1.0, "y": 0.0, "z": 0.0},
        {"x": 2.0, "y": 0.0, "z": 0.0},
    ],
    channels=[
        {"config": "DAPI", "exposure": 50},
        {"config": "FITC", "exposure": 100},
    ],
    stim_channels=[
        {"config": "Cy5", "exposure": 100},
    ],
    stim_frames=[1, 3],
)

events = list(seq)
print(f"{len(events)} events")

12 events


## 5. Run experiment

In [ ]:
from faro.core.controller import Controller

ctrl = Controller(mic, pipeline, writer=writer)
ctrl.run_experiment(events).wait()
ctrl.finish_experiment()

print("Experiment complete.")

## 6. Inspect the plate store

Note the difference from the slider layout:
- Root has **plate** metadata (rows, columns, wells)
- Each position is a **well** (`A/1/0/`, `A/2/0/`, `A/3/0/`)
- Each well has its own raw array `(t, c, y, x)` — no position dimension
- Each well has its own labels

In [7]:
import zarr

store = zarr.open(writer._zarr_path, mode="r")
print("Zarr store tree:")
print(store.tree())

Zarr store tree:


c:\Users\Alex\Programmierung\01_git\PhD\faro_main\.venv\Lib\site-packages\zarr\core\group.py:3535: ZarrUserWarning: Object at A is not recognized as a component of a Zarr hierarchy.
  warnings.warn(


/

In [8]:
# Plate metadata at root
import json

root_attrs = store.attrs.asdict()
ome = root_attrs.get("ome", root_attrs)
if "plate" in ome:
    print("Plate metadata:")
    print(json.dumps(ome["plate"], indent=2))
else:
    print("No plate metadata at root")

Plate metadata:
{
  "columns": [
    {
      "name": "1"
    },
    {
      "name": "2"
    },
    {
      "name": "3"
    }
  ],
  "rows": [
    {
      "name": "A"
    }
  ],
  "wells": [
    {
      "path": "A/1",
      "rowIndex": 0,
      "columnIndex": 0
    },
    {
      "path": "A/2",
      "rowIndex": 0,
      "columnIndex": 1
    },
    {
      "path": "A/3",
      "rowIndex": 0,
      "columnIndex": 2
    }
  ]
}


In [9]:
# Inspect well 1 (Pos0)
well_img = store["A/1/0"]
raw = well_img["0"]
print(f"Well A/1 raw: shape={raw.shape}, dtype={raw.dtype}")
print(f"  (t={raw.shape[0]}, c={raw.shape[1]}, y={raw.shape[2]}, x={raw.shape[3]})")

# Labels in this well
if "labels" in well_img:
    labels_grp = well_img["labels"]
    label_names = labels_grp.attrs.get("labels", [])
    print(f"Labels: {label_names}")
    for name in label_names:
        arr = labels_grp[f"{name}/0"]
        print(f"  {name}: shape={arr.shape}, dtype={arr.dtype}")

Well A/1 raw: shape=(4, 3, 512, 512), dtype=uint16
  (t=4, c=3, y=512, x=512)
Labels: ['stim_mask', 'particles', 'labels']
  stim_mask: shape=(4, 512, 512), dtype=uint8
  particles: shape=(4, 512, 512), dtype=int32
  labels: shape=(4, 512, 512), dtype=int32


In [10]:
# Compare: all wells have the same structure
for col in ["1", "2", "3"]:
    raw = store[f"A/{col}/0/0"]
    print(f"Well A/{col}: raw shape={raw.shape}")

Well A/1: raw shape=(4, 3, 512, 512)
Well A/2: raw shape=(4, 3, 512, 512)
Well A/3: raw shape=(4, 3, 512, 512)


In [11]:
# Tracking data
import pandas as pd

tracks_path = os.path.join(path, "tracks", "0_latest.parquet")
if os.path.exists(tracks_path):
    tracks = pd.read_parquet(tracks_path)
    print(
        f"Tracked {tracks['particle'].nunique()} cells across {tracks['timestep'].nunique()} frames"
    )
    tracks.head()
else:
    print("No tracking data found.")

Tracked 5 cells across 4 frames


## 7. Open in napari

Drag `acquisition.ome.zarr` into napari. The plate layout tiles all
positions spatially as a mosaic — each well is shown side by side.

Compare with the slider notebook where positions are a dimension slider.

> **Caveat:** napari-ome-zarr will only show the raw image channels as a
> tiled mosaic. The label layers (labels, particles, stim_mask) are stored
> in the zarr but **not auto-discovered** by the plate reader.
> To load them manually:
> ```python
> import zarr, napari
> viewer = napari.current_viewer()
> store = zarr.open("acquisition.ome.zarr", mode="r")
> viewer.add_labels(store["A/1/0/labels/labels/0"][:], name="labels (Pos0)")
> ```

In [12]:
print(f"Open in napari: {writer._zarr_path}")

Open in napari: C:\Users\Alex\AppData\Local\Temp\rtm-ome-zarr-plate-test\acquisition.ome.zarr
